# Stage 2 + CRC — GSM-Symbolic ladder, all six translators

**Project:** Certified Selective Reasoning · **Prompt version:** p1-v1 · CPU only.

Verifies **t1 t2 t3 t4 t5 t6** on **main / p1 / p2**, computes the full **15-pair ρ matrix**, and
runs conformal risk control over **three rule families**.

---

## Why three families rather than one

`ROLES` defines the rule family, and a k-of-6 rule is a *different* family from k-of-4. Its
floors and CRC thresholds cannot share a table with the SVAMP / GSM8K / GSM-Hard numbers.
So all three are computed and reported side by side:

| family | roles | purpose |
|---|---|---|
| **k4** | t1 t2 t3 t6 | **primary** — the P1 family, comparable with the other datasets |
| k5 | t1 t2 t3 t4 t6 | adds Mistral, drops the degenerate t5 |
| k6 | all six | complete picture; t5's abstention dominates the high-k end |

t4 earned its place: 69–74% `declares answer` across all three rungs, against 4–18% on the P1
datasets. Its retirement was a sample-size artefact. t5 did not: 44.0 → 40.8 → 36.1% with 33%
truncation on p2. It is verified and reported, and you should expect it to contribute almost
nothing above k=1.

## The family structure gets more interesting with six

```
t1  DeepSeek-Coder-V2-Lite    DeepSeek
t2  GLM-4-9B                  Zhipu
t3  Qwen2.5-Coder-7B          Alibaba  <- SAME FAMILY AS THE GENERATOR
t4  Mistral-7B                Mistral
t5  DeepSeek-V2-Lite-Chat     DeepSeek
t6  deepseek-coder-6.7b       DeepSeek
```

Fifteen pairs: **3 same-family** (all DeepSeek), **5 gen-family** (involving t3), **7
cross-family**. The P1 evidence had exactly one same-family pair with overlapping intervals
everywhere; this gives three, on three rungs, at 2500 rows per cell.

**This also sharpens a known caveat.** The note "k=2-of-4 can be satisfied by t1+t6, both
DeepSeek" gets worse with six roles: k=3 can be satisfied by t1+t5+t6, still all DeepSeek. A
certificate that reads as "three independent families agreed" would be false. CELL 6 measures
this directly — for each k, the share of gated problems whose agreeing set spans only one
family.

## Three things GSM-Symbolic forces

**Grouped splits.** `crc_core`'s docstring says it: *"a random split leaks templates across
calibration and test; this module does not implement one."* `crc_grouped.py` does. Both
guarantees are reported — within-template (random rows) and across-template (grouped) — and
the gap between them is template memorisation rather than generalisation.

**Cluster-aware bounds.** Measured ICC ≈ 0.5–0.6, design effect ≈ 29: 5000 rows carry about as
much information as 170 independent problems. Wilson is roughly 5× too narrow. It matters most
on zero-event cells, where rule-of-three on rows claims 0.15% and on templates 7.50%.

**The shared-50 restriction.** p2 covers 50 of main's 100 templates. Comparing ρ on 50 against
ρ on 100 confounds difficulty with composition — p1 generator accuracy is 76.1% on all 100 but
**72.1%** on the shared 50.

---

**Prerequisites:** `stage2_lib.py` and `crc_core.py` on disk (run CELL 2 and CELL 3 of
`stage2_verification_and_crc.ipynb`), and `raw_t{1..6}_gsm_symbolic_*.jsonl` for all three
splits in `CKPT_DIR`.

**Settled:** `extract_smt` already strips banned solver commands (`_BANNED_RE.sub("", body)`)
and records `+strippedN`. t3's ~80% banned-command rate never reaches Z3, so no gen-family ρ is
confounded by it.

## CELL 1 — Config

In [5]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================
CKPT_DIR = "./ckpt"
DATASETS = ["gsm_symbolic_main"]

# Every translator gets verified; rule FAMILIES decide what gets certified.
ROLES = ["t1", "t2", "t3", "t4", "t5", "t6"]

FAMILIES = {
    "k4": ["t1", "t2", "t3", "t6"],            # PRIMARY: matches the P1 runs
    "k5": ["t1", "t2", "t3", "t4", "t6"],      # + Mistral, - the degenerate t5
    "k6": ["t1", "t2", "t3", "t4", "t5", "t6"],
}
PRIMARY = "k4"          # the family whose numbers go in the main table

# Model family per translator. t3 shares Alibaba with the generator; t1/t5/t6
# are all DeepSeek, so there are THREE same-family pairs, not one.
FAM = {"t1": "DeepSeek", "t2": "Zhipu", "t3": "Alibaba",
       "t4": "Mistral", "t5": "DeepSeek", "t6": "DeepSeek"}
GEN_FAMILY = "Alibaba"                          # Qwen2.5-7B-Instruct

ALPHAS = [0.10, 0.05, 0.03, 0.02]
DELTA = 0.05
BONFERRONI = True
CAL_FRAC = 0.5
SEED = 0

AUDIT_TRIALS = 200
BOOT_B = 2000            # cluster bootstrap draws for a bound
BOOT_B_AUDIT = 500       # inner bootstrap inside the audit
BOOTSTRAP_RHO = 2000     # 15 pairs x 3 splits: raise only if you want the wait
Z3_TIMEOUT_MS = 10000

import os, sys, json, random, itertools
sys.path.insert(0, os.getcwd())
OUT_DIR = CKPT_DIR

for f in ("stage2_lib.py", "crc_core.py"):
    assert os.path.exists(f), (
        f"{f} not found. Run CELL 2 and CELL 3 of stage2_verification_and_crc.ipynb "
        f"first, or copy both files next to this notebook.")

PAIRS = list(itertools.combinations(ROLES, 2))
def pair_kind(a, b):
    if FAM[a] == FAM[b]:
        return "SAME-family"
    if GEN_FAMILY in (FAM[a], FAM[b]):
        return "gen-family"
    return "cross-family"

from collections import Counter
kinds = Counter(pair_kind(a, b) for a, b in PAIRS)
print(f"datasets  : {DATASETS}")
print(f"translators: {ROLES}")
print(f"families  : " + ", ".join(f"{k}=k-of-{len(v)}" for k, v in FAMILIES.items())
      + f"   primary {PRIMARY}")
print(f"pairs     : {len(PAIRS)}  " + ", ".join(f"{v} {k}" for k, v in sorted(kinds.items())))
print(f"alphas    : {ALPHAS}   delta {DELTA}   bonferroni {BONFERRONI}")

datasets  : ['gsm_symbolic_main']
translators: ['t1', 't2', 't3', 't4', 't5', 't6']
families  : k4=k-of-4, k5=k-of-5, k6=k-of-6   primary k4
pairs     : 15  3 SAME-family, 7 cross-family, 5 gen-family
alphas    : [0.1, 0.05, 0.03, 0.02]   delta 0.05   bonferroni True


## CELL 2 — The grouped-CRC module

Written to disk so it can be imported anywhere. Unchanged from the four-translator version —
every function takes the role list as an argument.

In [6]:
%%writefile crc_grouped.py
"""Cluster-aware CRC for GSM-Symbolic. Extends crc_core; does not replace it.

crc_core's own docstring says it: GSM-Symbolic has template structure, a random
row split leaks templates across calibration and test, and that module does not
implement a grouped split. This does.

--------------------------------------------------------------------------
WHY THE ROW-LEVEL PROCEDURE IS NOT WRONG, ONLY ANSWERING A DIFFERENT QUESTION

Split conformal needs calibration and test exchangeable. A random permutation of
ROWS delivers that, so the row-level bound is a valid guarantee - for a new
INSTANCE of one of these 100 templates. It says nothing about a template the
calibration set never saw, because 50 instances of one template are near
duplicates: same structure, different numbers.

Two guarantees, both real, reported side by side:

  within-template   random row split, Wilson bound.
                    "a fresh instance of a template we calibrated on"
  across-template   grouped split, cluster bootstrap bound.
                    "an instance of a template we have never seen"

The SECOND is the one a reader assumes you mean. The gap between them is the
part of the certificate that is template memorisation rather than generalisation,
and it is worth reporting as a number rather than a caveat.

--------------------------------------------------------------------------
THE BOUND ALSO HAS TO CHANGE, NOT JUST THE SPLIT

Wilson assumes n independent Bernoulli trials. With 50 instances per template
the rows are not independent, the effective sample size is nearer the template
count than the row count, and Wilson on n=2500 is far too tight. Here the
exchangeable unit is the TEMPLATE: resample templates with replacement, pool
within each draw, and take the (1-delta) percentile. Zero-event cells fall back
to rule-of-three on the number of GATED TEMPLATES, not gated rows - the
difference between a 0.12% and a 6% honest upper bound.
"""
from __future__ import annotations

import json
import os
import random
from collections import Counter, defaultdict

import crc_core as C


# ---------------------------------------------------------------------------
# GROUPS
# ---------------------------------------------------------------------------
def load_groups(ckpt_dir, dataset, pids, role="gen"):
    """pid -> template_id, read from meta written at Stage 1."""
    raw = C.load_raw(ckpt_dir, role, dataset)
    if raw is None:
        raise FileNotFoundError(f"no raw_{role}_{dataset}.jsonl in {ckpt_dir}")
    groups = {}
    for p in pids:
        meta = (raw[p] or {}).get("meta") or {}
        tid = meta.get("template_id")
        if tid is None:
            raise KeyError(f"{p} has no meta.template_id - regenerate with the "
                           f"GSM-Symbolic loader")
        groups[p] = tid
    return groups


def group_split(pids, groups, cal_frac, rng):
    """Split by TEMPLATE so no template appears in both halves."""
    gs = sorted({groups[p] for p in pids})
    rng.shuffle(gs)
    cut = int(len(gs) * cal_frac)
    cal_g = set(gs[:cut])
    cal = [p for p in pids if groups[p] in cal_g]
    tst = [p for p in pids if groups[p] not in cal_g]
    return cal, tst, len(cal_g), len(gs) - len(cal_g)


# ---------------------------------------------------------------------------
# CLUSTER-AWARE UPPER BOUND
# ---------------------------------------------------------------------------
def per_group(counts, correct, pids, groups, k):
    """template_id -> (n_gated, n_cw) at threshold k."""
    out = defaultdict(lambda: [0, 0])
    for p in pids:
        if counts[p] >= k:
            out[groups[p]][0] += 1
            if not correct[p]:
                out[groups[p]][1] += 1
    return {g: tuple(v) for g, v in out.items()}


def cluster_upper(pg, delta, B=2000, seed=0):
    """One-sided upper bound on the pooled CW rate, resampling TEMPLATES.

    pg: {template_id: (n_gated, n_cw)}. Returns (upper, n_rows, n_err,
    n_groups_gated, method).
    """
    gated = [(n, e) for n, e in pg.values() if n > 0]
    n_rows = sum(n for n, _ in gated)
    n_err = sum(e for _, e in gated)
    G = len(gated)
    if G == 0:
        return 1.0, 0, 0, 0, "no_coverage"
    if n_err == 0:
        # Rule of three on GATED TEMPLATES. Using gated ROWS here would claim a
        # bound ~50x tighter than the data can support.
        return min(1.0, 3.0 / G), n_rows, 0, G, "rule_of_three_groups"

    rng = random.Random(seed)
    draws = []
    for _ in range(B):
        num = den = 0
        for _ in range(G):
            n, e = gated[rng.randrange(G)]
            num += e
            den += n
        if den:
            draws.append(num / den)
    if not draws:
        return 1.0, n_rows, n_err, G, "bootstrap_failed"
    draws.sort()
    idx = min(len(draws) - 1, int(round((1.0 - delta) * len(draws))))
    return draws[idx], n_rows, n_err, G, "cluster_bootstrap"


def risk_at_k_grouped(counts, correct, pids, groups, k, delta, B=2000, seed=0):
    pg = per_group(counts, correct, pids, groups, k)
    up, n, err, G, method = cluster_upper(pg, delta, B, seed)
    return dict(k=k, n=n, err=err, rate=(err / n if n else 0.0), upper=up,
                n_groups=G, method=method,
                coverage=(n / len(pids) if pids else 0.0))


# ---------------------------------------------------------------------------
# SELECTION + AUDIT, GROUPED
# ---------------------------------------------------------------------------
def select_k_grouped(counts, correct, cal_pids, groups, alpha, delta, m,
                     bonferroni=True, B=2000, seed=0):
    """Smallest k (highest coverage) whose cluster bound is <= alpha."""
    d = delta / m if bonferroni else delta
    table, chosen = [], None
    for k in range(1, m + 1):
        row = risk_at_k_grouped(counts, correct, cal_pids, groups, k, d, B, seed)
        table.append(row)
        if chosen is None and row["n"] > 0 and row["upper"] <= alpha:
            chosen = k
    return chosen, {"table": table, "per_k_delta": d, "bonferroni": bonferroni}


def audit_grouped(counts, correct, pids, groups, alpha, delta, m,
                  trials=200, seed=0, bonferroni=True, cal_frac=0.5, B=500):
    """Repeat split-calibrate-test with GROUPED splits and count violations.

    Not part of the guarantee - the empirical test of it. Fewer trials and a
    smaller inner bootstrap than the row-level audit because each trial now
    costs a nested resample.
    """
    rng = random.Random(seed)
    picks, rates, covs, viol, abstain, skipped = [], [], [], 0, 0, 0
    for t in range(trials):
        cal, tst, ng_cal, ng_tst = group_split(pids, groups, cal_frac, rng)
        k, _ = select_k_grouped(counts, correct, cal, groups, alpha, delta, m,
                                bonferroni, B, seed + t)
        if k is None:
            abstain += 1
            continue
        picks.append(k)
        n, err, rate = C.risk_at_k(counts, correct, tst, k)
        if n == 0:
            skipped += 1
            continue
        rates.append(rate)
        covs.append(n / len(tst))
        if rate > alpha:
            viol += 1
    return {"trials": trials, "abstained": abstain, "selected": len(picks),
            "no_coverage_on_test": skipped,
            "k_distribution": dict(sorted(Counter(picks).items())),
            "mean_test_cw_rate": (sum(rates) / len(rates)) if rates else None,
            "mean_test_coverage": (sum(covs) / len(covs)) if covs else None,
            "violations": viol,
            "violation_rate": (viol / len(rates)) if rates else None,
            "target_violation_rate": delta}


# ---------------------------------------------------------------------------
# CLUSTER BOOTSTRAP FOR rho
# ---------------------------------------------------------------------------
def cluster_boot_rho(cert, correct, pids, groups, a, b, B=3000, seed=0):
    """rho between two translators' wrong-certification indicators.

    Resamples whole TEMPLATES. The row bootstrap in the base notebook treats 50
    instances of one template as 50 independent draws and reports an interval
    roughly sqrt(50) too narrow.
    """
    wrong = [p for p in pids if not correct[p]]
    if not wrong:
        return None, None, None, 0
    by_g = defaultdict(list)
    for p in wrong:
        by_g[groups[p]].append(p)
    gs = list(by_g)
    e1 = [1 if cert[a][p] else 0 for p in wrong]
    e2 = [1 if cert[b][p] else 0 for p in wrong]
    point = C_correlation(e1, e2)

    rng = random.Random(seed)
    out = []
    for _ in range(B):
        s1, s2 = [], []
        for _ in range(len(gs)):
            for p in by_g[gs[rng.randrange(len(gs))]]:
                s1.append(1 if cert[a][p] else 0)
                s2.append(1 if cert[b][p] else 0)
        r = C_correlation(s1, s2)
        if r is not None:
            out.append(r)
    if len(out) < B * 0.5:
        return point, None, None, len(wrong)
    out.sort()
    return (point, out[int(0.025 * len(out))], out[int(0.975 * len(out))],
            len(wrong))


def C_correlation(x, y):
    """Phi coefficient. Local copy so this module does not depend on import
    order of stage2_lib."""
    n = len(x)
    if n == 0:
        return None
    sx, sy = sum(x), sum(y)
    if sx in (0, n) or sy in (0, n):
        return None
    sxy = sum(a * b for a, b in zip(x, y))
    num = sxy / n - (sx / n) * (sy / n)
    den = ((sx / n) * (1 - sx / n) * (sy / n) * (1 - sy / n)) ** 0.5
    return num / den if den > 0 else None


# ---------------------------------------------------------------------------
# SHARED-TEMPLATE RESTRICTION
# ---------------------------------------------------------------------------
def shared_templates(ckpt_dir, datasets, role="gen"):
    """Template ids present in EVERY listed split.

    p2 covers 50 of main's 100 templates. Comparing rho measured on 50 against
    rho measured on 100 confounds difficulty with which problems are in the
    pool, which is the one thing this dataset exists to control.
    """
    sets = []
    for ds in datasets:
        raw = C.load_raw(ckpt_dir, role, ds)
        if raw is None:
            raise FileNotFoundError(f"no raw_{role}_{ds}.jsonl in {ckpt_dir}")
        sets.append({(r.get("meta") or {}).get("template_id") for r in raw.values()})
    out = set.intersection(*sets)
    out.discard(None)
    return sorted(out)


Overwriting crc_grouped.py


## CELL 3 — Self-test before trusting any number

Synthetic data with known template structure. Checks the three things that would silently
produce a wrong certificate.

In [7]:
# ============================================================
# CELL 3 — Self-test
# ============================================================
import importlib, crc_core as CRC, crc_grouped as G, stage2_lib as S
importlib.reload(CRC); importlib.reload(G)
import random as _r, statistics as _st

_rng = _r.Random(0)
_groups, _correct, _counts, _tp = {}, {}, {}, {}
for t in range(100):
    _tp[t] = 0.0 if _rng.random() < 0.75 else _rng.uniform(0.05, 0.60)
    for i in range(50):
        pid = f"p/{t:04d}_{i:02d}"
        _groups[pid] = t
        _counts[pid] = 4 if _rng.random() < 0.55 else _rng.randint(0, 3)
        _correct[pid] = not (_rng.random() < _tp[t])
_pids = list(_groups)

_cal, _tst, _gc, _gt = G.group_split(_pids, _groups, 0.5, _r.Random(1))
assert not ({_groups[p] for p in _cal} & {_groups[p] for p in _tst}), "split leaked templates"
print(f"[ok] grouped split: {_gc} cal / {_gt} test templates, 0 shared")

_cov = [p for p in _pids if _counts[p] >= 4]
_err = sum(1 for p in _cov if not _correct[p])
_w = S.wilson(_err, len(_cov))[1]
_c, _, _, _ng, _ = G.cluster_upper(G.per_group(_counts, _correct, _pids, _groups, 4),
                                   0.05, B=2000, seed=0)
assert _c >= _w, "cluster bound tighter than Wilson - clustering not modelled"
print(f"[ok] bound: Wilson {_w*100:.2f}% -> cluster {_c*100:.2f}% "
      f"over {_ng} templates ({_c/_w:.1f}x wider)")

_z, _, _, _gz, _mz = G.cluster_upper({t: (50, 0) for t in range(40)}, 0.05, B=200)
assert _mz == "rule_of_three_groups" and abs(_z - 3/40) < 1e-9
print(f"[ok] zero-event: rows would claim {3/2000*100:.2f}%, templates {_z*100:.2f}% "
      f"({_z/(3/2000):.0f}x apart)")

def _gap(fn, trials=150):
    d, r = [], _r.Random(1)
    for _ in range(trials):
        a, b = fn(r)
        na, _, ra = CRC.risk_at_k(_counts, _correct, a, 4)
        nb, _, rb = CRC.risk_at_k(_counts, _correct, b, 4)
        if na and nb: d.append(abs(ra - rb))
    return _st.mean(d)
_m1 = _gap(lambda r: (lambda s: (s[:len(s)//2], s[len(s)//2:]))(
    (lambda x: (r.shuffle(x), x)[1])(_pids[:])))
_m2 = _gap(lambda r: G.group_split(_pids, _groups, 0.5, r)[:2])
print(f"[ok] |cal-test| gap: rows {_m1*100:.2f}% vs grouped {_m2*100:.2f}% "
      f"(rows understate by {_m2/_m1:.1f}x)")
print("\nSELF-TEST PASSED")

[ok] grouped split: 50 cal / 50 test templates, 0 shared
[ok] bound: Wilson 9.15% -> cluster 11.05% over 100 templates (1.2x wider)
[ok] zero-event: rows would claim 0.15%, templates 7.50% (50x apart)
[ok] |cal-test| gap: rows 0.86% vs grouped 3.01% (rows understate by 3.5x)

SELF-TEST PASSED


## CELL 4 — Verify all six translators on all three splits

The only slow cell: Z3 over 6 roles × 12,500 rows. The verdict cache makes re-runs instant.
Counts are computed **per family** afterwards, so verification happens once.

In [8]:
# ============================================================
# CELL 4 — Verify. Cached per dataset.
# ============================================================
DATA = {}
for ds in DATASETS:
    print(f"\n=== {ds} ===")
    cache = os.path.join(CKPT_DIR, f"_verdict_cache6_{ds}.json")
    pids, correct, cert, models = CRC.verify_all(
        CKPT_DIR, ds, ROLES, timeout_ms=Z3_TIMEOUT_MS, cache=cache)
    groups = G.load_groups(CKPT_DIR, ds, pids)

    counts = {}
    for fam, roles in FAMILIES.items():
        counts[fam] = CRC.agreement_counts(cert, roles, pids)
        assert CRC.check_nested(counts[fam], pids, len(roles)), \
            f"{ds}/{fam}: rule family not nested - the conformal argument does not apply"

    nw = sum(1 for p in pids if not correct[p])
    ntpl = len(set(groups.values()))
    print(f"  n={len(pids)}  templates={ntpl}  instances/template={len(pids)/ntpl:.0f}")
    print(f"  generator accuracy {(1-nw/len(pids))*100:.1f}%  ({nw} wrong)")
    print(f"  nested: " + ", ".join(f"{f} ok" for f in FAMILIES))
    DATA[ds] = dict(pids=pids, correct=correct, cert=cert, models=models,
                    groups=groups, counts=counts, n_wrong=nw)

SHARED = G.shared_templates(CKPT_DIR, DATASETS)
print(f"\nshared templates across all three splits: {len(SHARED)}")
for ds in DATASETS:
    t = set(DATA[ds]["groups"].values())
    print(f"  {ds:22s} {len(t):3d} templates | {len(set(SHARED) & t)} shared")
assert SHARED, "no shared templates - the ladder comparison is not available"


=== gsm_symbolic_main ===
  t1  cov  49.9%  CW  52/2493   deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct
  t2  cov  38.2%  CW  33/1908   zai-org/GLM-4-9B-0414
  t3  cov  65.2%  CW  70/3262   Qwen/Qwen2.5-Coder-7B-Instruct
  t4  cov   5.5%  CW   8/273    mistralai/Mistral-7B-Instruct-v0.3
  t5  cov   0.5%  CW   0/23     deepseek-ai/DeepSeek-V2-Lite-Chat
  t6  cov  25.7%  CW  19/1284   deepseek-ai/deepseek-coder-6.7b-instruct
  n=5000  templates=100  instances/template=50
  generator accuracy 87.2%  (641 wrong)
  nested: k4 ok, k5 ok, k6 ok

shared templates across all three splits: 100
  gsm_symbolic_main      100 templates | 100 shared


## CELL 5 — Per-translator arms

All six, on the full splits. Both bounds shown: Wilson treats rows as independent, the cluster
bound resamples templates. Where they diverge, the cluster figure is the one that survives
review.

In [9]:
# ============================================================
# CELL 5 — Six arms per split
# ============================================================
ARMS = {}
for ds in DATASETS:
    D = DATA[ds]
    pids, correct, cert, groups = D["pids"], D["correct"], D["cert"], D["groups"]
    print(f"\n{'='*84}\n{ds}   n={len(pids)}   generator accuracy "
          f"{(1-D['n_wrong']/len(pids))*100:.1f}%\n{'='*84}")
    print(f"{'role':5s} {'family':10s} {'coverage':>9s} {'n_cov':>6s} {'CW':>4s} "
          f"{'CW rate':>8s} {'Wilson':>8s} {'cluster':>8s} {'tmpl':>5s}")
    ARMS[ds] = {}
    for r in ROLES:
        cov = [p for p in pids if cert[r][p]]
        cw = sum(1 for p in cov if not correct[p])
        w = S.wilson(cw, len(cov))[1] if cov else 1.0
        pg = {}
        for p in cov:
            g = groups[p]; n_, e_ = pg.get(g, (0, 0))
            pg[g] = (n_ + 1, e_ + (0 if correct[p] else 1))
        c, _, _, ng, meth = G.cluster_upper(pg, DELTA, B=BOOT_B, seed=SEED)
        print(f"{r:5s} {FAM[r]:10s} {len(cov)/len(pids)*100:8.1f}% {len(cov):6d} {cw:4d} "
              f"{cw/max(len(cov),1)*100:7.2f}% {w*100:7.2f}% {c*100:7.2f}% {ng:5d}")
        ARMS[ds][r] = dict(coverage=len(cov)/len(pids), n_cov=len(cov), cw=cw,
                           cw_rate=cw/max(len(cov), 1), wilson=w, cluster=c,
                           n_groups=ng, method=meth)


gsm_symbolic_main   n=5000   generator accuracy 87.2%
role  family      coverage  n_cov   CW  CW rate   Wilson  cluster  tmpl
t1    DeepSeek       49.9%   2493   52    2.09%    2.72%    5.11%    95
t2    Zhipu          38.2%   1908   33    1.73%    2.42%    4.58%    86
t3    Alibaba        65.2%   3262   70    2.15%    2.70%    4.27%    96
t4    Mistral         5.5%    273    8    2.93%    5.67%    8.60%    47
t5    DeepSeek        0.5%     23    0    0.00%   14.31%   20.00%    15
t6    DeepSeek       25.7%   1284   19    1.48%    2.30%    3.84%    74


## CELL 6 — Risk curves per family, and the family-diversity audit

The second table is the one that matters for how the certificate can be *described*. With three
DeepSeek translators in the pool, an agreeing set of size 3 can be t1+t5+t6 — all one family.
Calling that "three independent families agreed" would be false.

`single-family` is the share of gated problems whose agreeing set spans exactly one model
family. If it is material at your selected k, either restrict the roles or state the guarantee
as operating over *agreement count* irrespective of family.

In [10]:
# ============================================================
# CELL 6 — Risk curves and family diversity
# ============================================================
CURVES, FLOORS = {}, {}
for ds in DATASETS:
    D = DATA[ds]
    pids, correct, cert, groups = D["pids"], D["correct"], D["cert"], D["groups"]
    print(f"\n{'#'*84}\n# {ds}\n{'#'*84}")
    CURVES[ds], FLOORS[ds] = {}, {}
    for fam, roles in FAMILIES.items():
        counts = D["counts"][fam]
        print(f"\n  {fam} = k-of-{len(roles)} over {roles}")
        print(f"  {'k':>3s} {'n_gated':>8s} {'coverage':>9s} {'tmpl':>5s} {'CW':>4s} "
              f"{'CW rate':>8s} {'Wilson':>8s} {'cluster':>8s} {'1-family':>9s} {'fams':>5s}")
        curve = []
        for k in range(1, len(roles) + 1):
            row = G.risk_at_k_grouped(counts, correct, pids, groups, k, DELTA,
                                      B=BOOT_B, seed=SEED)
            row["wilson_upper"] = S.wilson(row["err"], row["n"])[1] if row["n"] else 1.0
            gated = [p for p in pids if counts[p] >= k]
            nfams = [len({FAM[r] for r in roles if cert[r][p]}) for p in gated]
            single = sum(1 for x in nfams if x <= 1) / max(len(gated), 1)
            row["single_family_share"] = single
            row["mean_families"] = sum(nfams) / max(len(nfams), 1)
            curve.append(row)
            flag = "  <--" if single > 0.10 and row["n"] else ""
            print(f"  {k:3d} {row['n']:8d} {row['coverage']*100:8.1f}% {row['n_groups']:5d} "
                  f"{row['err']:4d} {row['rate']*100:7.2f}% {row['wilson_upper']*100:7.2f}% "
                  f"{row['upper']*100:7.2f}% {single*100:8.1f}% "
                  f"{row['mean_families']:5.2f}{flag}")
        CURVES[ds][fam] = curve
        FLOORS[ds][fam] = curve[-1]
        f = curve[-1]
        print(f"  FLOOR({fam}): unanimous {len(roles)}-of-{len(roles)} still admits "
              f"{f['rate']*100:.2f}% CW (cluster upper {f['upper']*100:.2f}%, "
              f"{f['n_groups']} templates)")
print("\n'1-family' = share of gated problems where every agreeing translator is from")
print("the SAME model family. A high value at your selected k means the certificate")
print("cannot be described as cross-family agreement.")


####################################################################################
# gsm_symbolic_main
####################################################################################

  k4 = k-of-4 over ['t1', 't2', 't3', 't6']
    k  n_gated  coverage  tmpl   CW  CW rate   Wilson  cluster  1-family  fams
    1     3784     75.7%   100   96    2.54%    3.09%    4.63%     27.2%  2.10  <--
    2     2799     56.0%    89   41    1.46%    1.98%    3.92%      1.6%  2.49
    3     1749     35.0%    73   28    1.60%    2.30%    4.81%      0.0%  2.81
    4      615     12.3%    51    9    1.46%    2.76%    4.35%      0.0%  3.00
  FLOOR(k4): unanimous 4-of-4 still admits 1.46% CW (cluster upper 4.35%, 51 templates)

  k5 = k-of-5 over ['t1', 't2', 't3', 't4', 't6']
    k  n_gated  coverage  tmpl   CW  CW rate   Wilson  cluster  1-family  fams
    1     3787     75.7%   100   96    2.53%    3.09%    4.63%     26.5%  2.17  <--
    2     2824     56.5%    89   41    1.45%    1.96%    3.88%

## CELL 7 — The ladder, restricted to the shared templates

Same 50 templates at every rung, 2500 rows per cell. Only the clause count changes.

Pre-registered before any ρ was computed: median integer in the question falls **16 → 14 → 13**,
p90 falls **250 → 150 → 130**. Numbers get *smaller* as difficulty rises, so a rising ρ cannot
be attributed to larger integers stressing Int/Real handling.

In [11]:
# ============================================================
# CELL 7 — Ladder on the shared templates
# ============================================================
SH = set(SHARED)
LADDER = {}
print(f"restricted to {len(SH)} shared templates\n")
for fam, roles in FAMILIES.items():
    print(f"--- {fam} = k-of-{len(roles)} ---")
    print(f"{'split':22s} {'rows':>6s} {'gen acc':>8s}" +
          "".join(f"{'k='+str(k):>10s}" for k in range(1, len(roles)+1)) + "   coverage")
    for ds in DATASETS:
        D = DATA[ds]
        sub = [p for p in D["pids"] if D["groups"][p] in SH]
        nw = sum(1 for p in sub if not D["correct"][p])
        cells = [G.risk_at_k_grouped(D["counts"][fam], D["correct"], sub, D["groups"],
                                     k, DELTA, B=BOOT_B, seed=SEED)
                 for k in range(1, len(roles)+1)]
        LADDER.setdefault(ds, {})[fam] = dict(sub=sub, n=len(sub),
                                              acc=1-nw/len(sub), cells=cells)
        print(f"{ds:22s} {len(sub):6d} {(1-nw/len(sub))*100:7.1f}%" +
              "".join(f"{c['coverage']*100:9.1f}%" for c in cells))
    print(f"{'':22s} {'':6s} {'':8s}" +
          "".join(f"{'':>10s}" for _ in roles) + "   CW rate")
    for ds in DATASETS:
        print(f"{ds:22s} {'':6s} {'':8s}" +
              "".join(f"{c['rate']*100:9.2f}%" for c in LADDER[ds][fam]["cells"]))
    print()

restricted to 100 shared templates

--- k4 = k-of-4 ---
split                    rows  gen acc       k=1       k=2       k=3       k=4   coverage
gsm_symbolic_main        5000    87.2%     75.7%     56.0%     35.0%     12.3%
                                                                                 CW rate
gsm_symbolic_main                          2.54%     1.46%     1.60%     1.46%

--- k5 = k-of-5 ---
split                    rows  gen acc       k=1       k=2       k=3       k=4       k=5   coverage
gsm_symbolic_main        5000    87.2%     75.7%     56.5%     36.3%     14.3%      1.6%
                                                                                           CW rate
gsm_symbolic_main                          2.53%     1.45%     1.65%     1.96%     1.28%

--- k6 = k-of-6 ---
split                    rows  gen acc       k=1       k=2       k=3       k=4       k=5       k=6   coverage
gsm_symbolic_main        5000    87.2%     75.7%     56.5%     36.4%     14.4%

## CELL 8 — The full 15-pair ρ matrix

ρ is estimated on the wrong-answer subset only: that is the only place a translator can wrongly
certify, so ε_i is exactly the single-translator failure rate the decorrelation argument is
written in terms of.

**The bootstrap resamples whole templates.** A row bootstrap treats 50 instances of one template
as 50 independent draws and returns an interval roughly √50 too narrow.

Three same-family pairs (all DeepSeek), five gen-family (involving t3/Alibaba), seven
cross-family. The P1 evidence had one same-family pair with overlapping intervals on every
dataset; this is the power to settle it.

In [12]:
# ============================================================
# CELL 8 — 15-pair rho matrix, cluster bootstrap
# ============================================================
RHO = {}
for ds in DATASETS:
    D = DATA[ds]
    sub = LADDER[ds][PRIMARY]["sub"]
    wrong = [p for p in sub if not D["correct"][p]]
    ntw = len({D["groups"][p] for p in wrong})
    print(f"\n{'='*88}\n{ds}   wrong-answer pool {len(wrong)} over {ntw} templates\n{'='*88}")
    print(f"{'pair':9s} {'families':24s} {'kind':13s} {'rho':>7s} "
          f"{'cluster 95% CI':>18s} {'row 95% CI':>18s}")
    rows = []
    for a, b in PAIRS:
        pt, lo, hi, nw = G.cluster_boot_rho(D["cert"], D["correct"], sub, D["groups"],
                                            a, b, B=BOOTSTRAP_RHO, seed=SEED)
        e1 = [1 if D["cert"][a][p] else 0 for p in wrong]
        e2 = [1 if D["cert"][b][p] else 0 for p in wrong]
        _rr = random.Random(SEED); rr = []
        for _ in range(BOOTSTRAP_RHO):
            idx = [_rr.randrange(len(e1)) for _ in range(len(e1))]
            v = S.correlation([e1[i] for i in idx], [e2[i] for i in idx])
            if v is not None: rr.append(v)
        rr.sort()
        rlo, rhi = (rr[int(.025*len(rr))], rr[int(.975*len(rr))]) if len(rr) > 100 else (None, None)
        kind = pair_kind(a, b)
        cci = f"[{lo:+.2f},{hi:+.2f}]" if lo is not None else "undefined"
        rci = f"[{rlo:+.2f},{rhi:+.2f}]" if rlo is not None else "undefined"
        span = "  spans 0" if (lo is not None and lo < 0 < hi) else ""
        print(f"{a}x{b:6s} {FAM[a]+' x '+FAM[b]:24s} {kind:13s} "
              f"{('    n/a' if pt is None else f'{pt:+7.3f}')} {cci:>18s} {rci:>18s}{span}")
        rows.append(dict(pair=f"{a}x{b}", a=a, b=b, kind=kind, rho=pt,
                         ci_lo=lo, ci_hi=hi, row_ci_lo=rlo, row_ci_hi=rhi, n_wrong=nw))
    RHO[ds] = rows

print(f"\n{'='*88}\nMEAN rho BY PAIR KIND, ACROSS THE LADDER\n{'='*88}")
print(f"{'kind':13s} {'pairs':>6s} " +
      "".join(f"{ds.replace('gsm_symbolic_',''):>12s}" for ds in DATASETS))
for kind in ["SAME-family", "gen-family", "cross-family"]:
    cells = []
    npair = 0
    for ds in DATASETS:
        vals = [r["rho"] for r in RHO[ds] if r["kind"] == kind and r["rho"] is not None]
        npair = max(npair, len(vals))
        cells.append(f"{sum(vals)/len(vals):+.3f}" if vals else "n/a")
    print(f"{kind:13s} {npair:6d} " + "".join(f"{c:>12s}" for c in cells))

print(f"\n{'='*88}\nEVERY SAME-FAMILY PAIR vs THE DEPLOYED CROSS-FAMILY RULE\n{'='*88}")
print(f"{'pair':9s} {'kind':13s} " +
      "".join(f"{ds.replace('gsm_symbolic_',''):>24s}" for ds in DATASETS))
for pair in [r["pair"] for r in RHO[DATASETS[0]]
             if r["kind"] == "SAME-family"] + ["t1xt2"]:
    cells = []
    for ds in DATASETS:
        r = next((x for x in RHO[ds] if x["pair"] == pair), None)
        cells.append("n/a" if not r or r["rho"] is None
                     else f"{r['rho']:+.3f} [{r['ci_lo']:+.2f},{r['ci_hi']:+.2f}]")
    k = next((x["kind"] for x in RHO[DATASETS[0]] if x["pair"] == pair), "")
    print(f"{pair:9s} {k:13s} " + "".join(f"{c:>24s}" for c in cells))
print("\nP1 reference: rho(t1,t6) same-family 0.276/0.187/0.372 vs rho(t1,t2)")
print("cross-family 0.248/0.411/0.580 on SVAMP/GSM8K/GSM-Hard - same-family LOWER")
print("on both well-powered datasets, every interval overlapping.")


gsm_symbolic_main   wrong-answer pool 641 over 60 templates
pair      families                 kind              rho     cluster 95% CI         row 95% CI
t1xt2     DeepSeek x Zhipu         cross-family   +0.681      [-0.01,+0.79]      [+0.55,+0.78]  spans 0
t1xt3     DeepSeek x Alibaba       gen-family     +0.482      [-0.04,+0.76]      [+0.37,+0.59]  spans 0
t1xt4     DeepSeek x Mistral       cross-family   +0.378      [+0.34,+0.43]      [+0.23,+0.50]
t1xt5     DeepSeek x DeepSeek      SAME-family       n/a          undefined          undefined
t1xt6     DeepSeek x DeepSeek      SAME-family    +0.453      [-0.01,+0.55]      [+0.31,+0.59]  spans 0
t2xt3     Zhipu x Alibaba          gen-family     +0.462      [-0.02,+0.67]      [+0.33,+0.58]  spans 0
t2xt4     Zhipu x Mistral          cross-family   +0.292      [+0.26,+0.32]      [+0.08,+0.47]
t2xt5     Zhipu x DeepSeek         cross-family      n/a          undefined          undefined
t2xt6     Zhipu x DeepSeek         cross-family 

## CELL 9 — Conformal risk control, every family, both guarantees

For each split, family and α, two certificates:

- **within-template**, random row split + Wilson — a fresh instance of a calibrated template
- **across-template**, grouped split + cluster bound — an instance of an unseen template

The audit repeats the whole split-calibrate-test procedure and counts violations. It is not
part of the guarantee; it is the test of whether the guarantee is delivered. A violation rate
materially above δ means the procedure is anticonservative and the bound is wrong.

Bonferroni divides δ by the number of candidate thresholds, so k6 pays δ/6 against k4's δ/4 —
more roles is not free.

In [13]:
# ============================================================
# CELL 9 — CRC per family
# ============================================================
RESULTS = {}
for ds in DATASETS:
    D = DATA[ds]
    pids, correct, groups = D["pids"], D["correct"], D["groups"]
    print(f"\n{'#'*84}\n# {ds}   n={len(pids)}   "
          f"{len(set(groups.values()))} templates\n{'#'*84}")

    rng = random.Random(SEED); sh = pids[:]; rng.shuffle(sh)
    cut = int(len(sh) * CAL_FRAC); cal_r, tst_r = sh[:cut], sh[cut:]
    cal_g, tst_g, ng_c, ng_t = G.group_split(pids, groups, CAL_FRAC, random.Random(SEED))
    print(f"row split |cal|={len(cal_r)} |test|={len(tst_r)}   "
          f"grouped |cal|={len(cal_g)} ({ng_c} tmpl) |test|={len(tst_g)} ({ng_t} tmpl)")

    entry = {"dataset": ds, "n": len(pids), "n_templates": len(set(groups.values())),
             "models": D["models"], "families": FAMILIES, "alphas": {}}
    for fam, roles in FAMILIES.items():
        counts, m = D["counts"][fam], len(roles)
        print(f"\n  --- {fam} = k-of-{m}  (Bonferroni delta/{m} = {DELTA/m:.4f}) ---")
        for alpha in ALPHAS:
            k_r, _ = CRC.select_k(counts, correct, cal_r, alpha, DELTA, m,
                                  "wilson", BONFERRONI)
            within = {"selected_k": None}
            wtxt = "abstain"
            if k_r is not None:
                ev = CRC.evaluate(counts, correct, tst_r, k_r)
                within = {"selected_k": k_r, "test": ev}
                wtxt = (f"k={k_r} cov {ev['coverage']*100:.1f}% CW {ev['cw_rate']*100:.2f}% "
                        f"{'HOLDS' if ev['cw_rate'] <= alpha else 'VIOLATED'}")

            k_g, diag = G.select_k_grouped(counts, correct, cal_g, groups, alpha, DELTA,
                                           m, BONFERRONI, BOOT_B, SEED)
            across = {"selected_k": None}
            if k_g is None:
                best = min((r["upper"] for r in diag["table"] if r["n"] > 0), default=1.0)
                across = {"selected_k": None, "tightest_bound": best}
                gtxt = f"abstain (tightest {best*100:.2f}%)"
            else:
                n_, e_, r_ = CRC.risk_at_k(counts, correct, tst_g, k_g)
                across = {"selected_k": k_g, "calibration": diag,
                          "test": {"n_covered": n_, "cw": e_, "cw_rate": r_,
                                   "coverage": n_/len(tst_g)}}
                gtxt = (f"k={k_g} cov {n_/len(tst_g)*100:.1f}% CW {r_*100:.2f}% "
                        f"{'HOLDS' if r_ <= alpha else 'VIOLATED'}")

            au = G.audit_grouped(counts, correct, pids, groups, alpha, DELTA, m,
                                 AUDIT_TRIALS, SEED, BONFERRONI, CAL_FRAC, BOOT_B_AUDIT)
            across["audit"] = au
            if au["violation_rate"] is None:
                atxt = f"abstained {au['abstained']}/{au['trials']}"
            else:
                atxt = (f"viol {au['violation_rate']*100:.1f}% (<={DELTA*100:.0f}%)"
                        + ("  ANTICONSERVATIVE" if au["violation_rate"] > DELTA*1.5 else ""))
            print(f"    a={alpha*100:2.0f}%  within: {wtxt:42s}")
            print(f"           across: {gtxt:42s} audit {atxt}")
            entry["alphas"].setdefault(fam, {})[str(alpha)] = {
                "within_template": within, "across_template": across}

    entry["risk_curves"] = CURVES[ds]
    entry["floors"] = FLOORS[ds]
    entry["arms"] = ARMS[ds]
    entry["rho"] = RHO[ds]
    entry["shared_templates"] = SHARED
    entry["ladder"] = {f: {"n": LADDER[ds][f]["n"], "acc": LADDER[ds][f]["acc"],
                           "cells": LADDER[ds][f]["cells"]} for f in FAMILIES}
    RESULTS[ds] = entry
    out = os.path.join(OUT_DIR, f"crc6_{ds}.json")
    json.dump(entry, open(out, "w", encoding="utf-8"), indent=2, default=float)
    print(f"\n  written to {out}")


####################################################################################
# gsm_symbolic_main   n=5000   100 templates
####################################################################################
row split |cal|=2500 |test|=2500   grouped |cal|=2500 (50 tmpl) |test|=2500 (50 tmpl)

  --- k4 = k-of-4  (Bonferroni delta/4 = 0.0125) ---
    a=10%  within: k=1 cov 75.8% CW 2.22% HOLDS              
           across: k=1 cov 73.9% CW 0.81% HOLDS               audit viol 0.0% (<=5%)
    a= 5%  within: k=1 cov 75.8% CW 2.22% HOLDS              
           across: abstain (tightest 8.94%)                   audit viol 0.0% (<=5%)
    a= 3%  within: k=2 cov 54.8% CW 1.17% HOLDS              
           across: abstain (tightest 8.94%)                   audit viol 30.9% (<=5%)  ANTICONSERVATIVE
    a= 2%  within: abstain                                   
           across: abstain (tightest 8.94%)                   audit viol 100.0% (<=5%)  ANTICONSERVATIVE

  --- k5 = k-of-

## CELL 10 — The sentences for the paper

Reports, per family, the tightest α that both selects a threshold and passes the audit. An α
that selects but violates is not a result.

In [14]:
# ============================================================
# CELL 10 — Paper sentences
# ============================================================
print("="*84)
for ds in DATASETS:
    e = RESULTS[ds]
    print(f"\n{ds}  (n={e['n']}, {e['n_templates']} templates)")
    for fam, roles in FAMILIES.items():
        fl = e["floors"][fam]
        best = None
        for alpha in sorted(ALPHAS):
            a = e["alphas"][fam][str(alpha)]["across_template"]
            au = a.get("audit", {})
            if a["selected_k"] is not None and au.get("violation_rate") is not None \
                    and au["violation_rate"] <= DELTA * 1.5:
                best = (alpha, a); break
        tag = " [PRIMARY]" if fam == PRIMARY else ""
        if best is None:
            print(f"  {fam} k-of-{len(roles)}{tag}: no target below "
                  f"{fl['upper']*100:.1f}% is certifiable across UNSEEN templates.")
            print(f"      Unanimous agreement still admits {fl['rate']*100:.2f}% CW "
                  f"(cluster upper {fl['upper']*100:.2f}%, {fl['n_groups']} templates),")
            print(f"      which lower-bounds any distribution-free certificate on this family.")
        else:
            alpha, a = best; au = a["audit"]; ev = a["test"]
            sf = next((c["single_family_share"] for c in CURVES[ds][fam]
                       if c["k"] == a["selected_k"]), 0.0)
            print(f"  {fam} k-of-{len(roles)}{tag}: CW <= {alpha*100:.0f}% at "
                  f"{(1-DELTA)*100:.0f}% confidence, k={a['selected_k']}, "
                  f"coverage {ev['coverage']*100:.1f}%,")
            print(f"      {au['violation_rate']*100:.1f}% violations over {au['trials']} "
                  f"GROUPED splits (Bonferroni delta/{len(roles)}).")
            print(f"      Calibration and test templates are disjoint, so the guarantee")
            print(f"      extends to problem templates never seen at calibration.")
            if sf > 0.10:
                print(f"      CAVEAT: {sf*100:.0f}% of gated problems are agreed by a "
                      f"SINGLE model family -")
                print(f"      describe this as agreement count, not cross-family agreement.")

print("\n" + "="*84)
print("MEMORISATION GAP - within-template minus across-template, primary family")
print("="*84)
print(f"{'split':22s} {'alpha':>7s} {'within k':>9s} {'across k':>9s} "
      f"{'within cov':>11s} {'across cov':>11s}")
for ds in DATASETS:
    for alpha in ALPHAS:
        a = RESULTS[ds]["alphas"][PRIMARY][str(alpha)]
        w, g = a["within_template"], a["across_template"]
        wc = f"{w['test']['coverage']*100:.1f}%" if w["selected_k"] else "-"
        gc = f"{g['test']['coverage']*100:.1f}%" if g["selected_k"] else "-"
        print(f"{ds:22s} {alpha*100:6.0f}% {str(w['selected_k'] or 'abstain'):>9s} "
              f"{str(g['selected_k'] or 'abstain'):>9s} {wc:>11s} {gc:>11s}")
print("\nWhere within-template certifies and across-template abstains, the difference")
print("is template memorisation, not verification quality.")

spans = [(ds, r["pair"], r["kind"]) for ds in DATASETS for r in RHO[ds]
         if r["ci_lo"] is not None and r["ci_lo"] < 0 < r["ci_hi"]]
if spans:
    print(f"\nCAVEAT: {len(spans)}/{len(PAIRS)*len(DATASETS)} rho intervals span zero under")
    print("the cluster bootstrap. Do not present those point estimates as measured:")
    for ds, p, k in spans[:8]:
        print(f"  {ds.replace('gsm_symbolic_',''):6s} {p:9s} {k}")


gsm_symbolic_main  (n=5000, 100 templates)
  k4 k-of-4 [PRIMARY]: CW <= 10% at 95% confidence, k=1, coverage 73.9%,
      0.0% violations over 200 GROUPED splits (Bonferroni delta/4).
      Calibration and test templates are disjoint, so the guarantee
      extends to problem templates never seen at calibration.
      CAVEAT: 27% of gated problems are agreed by a SINGLE model family -
      describe this as agreement count, not cross-family agreement.
  k5 k-of-5: CW <= 10% at 95% confidence, k=2, coverage 52.2%,
      0.0% violations over 200 GROUPED splits (Bonferroni delta/5).
      Calibration and test templates are disjoint, so the guarantee
      extends to problem templates never seen at calibration.
  k6 k-of-6: CW <= 10% at 95% confidence, k=2, coverage 52.2%,
      0.0% violations over 200 GROUPED splits (Bonferroni delta/6).
      Calibration and test templates are disjoint, so the guarantee
      extends to problem templates never seen at calibration.

MEMORISATION GAP - w

## CELL 11 — CW cases for adjudication

**Write the adjudication protocol before opening this cell.** Three SVAMP CW cases were already
inspected and all three look like benchmark label noise rather than model error, so that
inspection has to be declared post-hoc. Do not add to the problem.

Cases are grouped by template: if one template produces most of the CW, that is a property of
one problem type, not a general failure rate.

In [15]:
# ============================================================
# CELL 11 — CW cases, grouped by template
# ============================================================
FAM_REPORT = PRIMARY
K_REPORT = len(FAMILIES[FAM_REPORT])       # unanimous: the cases defining the floor

for ds in DATASETS:
    D = DATA[ds]
    counts = D["counts"][FAM_REPORT]
    cw = [p for p in D["pids"] if counts[p] >= K_REPORT and not D["correct"][p]]
    by_t = {}
    for p in cw:
        by_t.setdefault(D["groups"][p], []).append(p)
    print(f"\n{ds}: {len(cw)} CW at {FAM_REPORT} k={K_REPORT}, across {len(by_t)} templates")
    if not cw:
        print("  none - the floor is a zero-event cell; see the rule-of-three bound above.")
        continue
    for t, ps in sorted(by_t.items(), key=lambda kv: -len(kv[1]))[:5]:
        print(f"  template {t:4d}: {len(ps):3d} cases   e.g. {ps[0]}")
    top = max(len(v) for v in by_t.values())
    if top / len(cw) > 0.4:
        print(f"  NOTE: {top/len(cw)*100:.0f}% of CW comes from a single template. The")
        print(f"  pooled rate describes one problem type, not the family as a whole.")

    out = os.path.join(OUT_DIR, f"cw_cases_{ds}_{FAM_REPORT}_k{K_REPORT}.jsonl")
    raw_gen = CRC.load_raw(CKPT_DIR, "gen", ds)
    with open(out, "w", encoding="utf-8") as f:
        for p in cw:
            certd = [r for r in FAMILIES[FAM_REPORT] if D["cert"][r][p]]
            f.write(json.dumps({
                "pid": p, "template_id": D["groups"][p],
                "question": raw_gen[p]["question"], "gold": raw_gen[p]["gold"],
                "generator_output": raw_gen[p]["outputs"][0],
                "certified_by": certd,
                "families": sorted({FAM[r] for r in certd}),
                "all_six": {r: bool(D["cert"][r][p]) for r in ROLES},
            }, ensure_ascii=False) + "\n")
    print(f"  -> {out}")


gsm_symbolic_main: 9 CW at k4 k=4, across 1 templates
  template   76:   9 cases   e.g. gsm_symbolic_main/0076_02
  NOTE: 100% of CW comes from a single template. The
  pooled rate describes one problem type, not the family as a whole.
  -> ./ckpt\cw_cases_gsm_symbolic_main_k4_k4.jsonl
